# 05. MAF でシングルエージェントを構築する (Agent Skills あり)

## コードの解説

**Agent Skills** は、エージェントにモジュラーな知識を提供する仕組みです。
`SkillsProvider` が `skills/` ディレクトリの `SKILL.md` を自動探索し、
**Progressive Disclosure パターン** でエージェントに提供します。

### Progressive Disclosure パターン（3 段階）

```
1. Advertise（広告）
   └─ スキル名 + 説明だけをシステムプロンプトに注入（軽量・低トークン）

2. Load（ロード）
   └─ エージェントが load_skill ツールを呼び出し → SKILL.md 本文を取得

3. Read Resources（リソース読込）
   └─ エージェントが read_skill_resource ツールで FAQ/テンプレート等を取得
```

### 従来のアプローチとの比較

| 項目 | 従来（instructions に全部書く） | Agent Skills |
|-----|----------------------------|--------------|
| コンテキスト使用量 | 大きい（常に全知識をロード） | 小さい（必要時のみロード） |
| 知識の更新 | コード修正が必要 | `SKILL.md` を更新するだけ |
| スケーラビリティ | スキルが増えると破綻 | スキルを追加するだけ |

### ディレクトリ構造

```
skills/
├── travel-planner/
│   ├── SKILL.md              ← メインの指示書（YAML frontmatter + 本文）
│   ├── assets/
│   │   └── itinerary-template.md
│   └── references/
│       └── TRAVEL_FAQ.md
└── code-reviewer/
    ├── SKILL.md
    └── references/
        └── CHECKLIST.md
```

### SKILL.md のフォーマット

```markdown
---
name: travel-planner
description: 旅行の計画と予約に関するガイダンスを提供します。
---

# 本文（エージェント向け詳細指示）
ここに詳細なドメイン知識を記述...
```

In [ ]:
import os
from dotenv import load_dotenv
from agent_framework import Agent, SkillsProvider
from agent_framework.openai import OpenAIChatCompletionClient

load_dotenv()

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY  = os.getenv("AZURE_OPENAI_API_KEY")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

USE_KEY_AUTH = os.getenv("USE_KEY_AUTH", "False").lower() in ("true", "1", "t")
if USE_KEY_AUTH:
    auth_kwargs = dict(api_key=AZURE_OPENAI_API_KEY)
else:
    from azure.identity.aio import DefaultAzureCredential
    os.environ.pop("AZURE_OPENAI_API_KEY", None)
    auth_kwargs = dict(credential=DefaultAzureCredential())


# ---------------------------------------------------------------
# SkillsProvider の作成
# skill_paths に skills/ ディレクトリを指定すると、
# 配下の SKILL.md を自動探索してスキルとして登録する
# ---------------------------------------------------------------
skills_provider = SkillsProvider(skill_paths="skills")
print("SkillsProvider 作成完了")


# ---------------------------------------------------------------
# Agent Skills 付きエージェントの作成
#
# ポイント: context_providers=[skills_provider] で渡す
#   → エージェントは load_skill / read_skill_resource ツールを自動的に使い、
#     必要なスキルの詳細をオンデマンドでロードする
# ---------------------------------------------------------------
agent = Agent(
    client=OpenAIChatCompletionClient(
        **auth_kwargs,
        azure_endpoint=AZURE_OPENAI_ENDPOINT,
        model=MODEL_DEPLOYMENT_NAME,
    ),
    name="TravelPlannerAgent",
    instructions=(
        "あなたは旅行計画の専門家です。"
        "スキルを活用して、最適な旅行プランをユーザーに提案してください。"
    ),
    context_providers=[skills_provider],
)

# エージェントの実行
response = await agent.run("3 泊 4 日のパリ旅行プランを作ってください。")
print(response)
